# RetailHero — cross-fitting проверка uplift-модели

**Цель:** оценить устойчивость uplift-таргетинга на 3 фолдах, сравнить его со случайной рассылкой и простыми правилами RFM. Каждый клиент оценивается моделью, которая не обучалась на его результате.

**Внимание:** данные уже использовались при разработке модели — это ретроспективная проверка устойчивости, а не полностью независимая финальная оценка и не новый A/B-тест.

Фиксируем до запуска: 3 фолда, seed 20261030, HistGradientBoostingClassifier, топ-10% по ранжированию, 2000 bootstrap повторов; главный baseline — расходы клиента до рекламы.

In [ ]:
%pip -q install "pandas>=2" "numpy>=1.24" "scikit-learn>=1.3" "duckdb>=1.1" "huggingface_hub>=0.20"

In [ ]:
from pathlib import Path
import subprocess, sys

PROJECT = Path('/content/retailhero_cv_repo')
BRANCH = 'validation/crossfit-stability-20261030'
URL = 'https://github.com/venfun77/retailhero-ab-uplift.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', BRANCH,
                    URL, str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
print('Код проверки загружен:', PROJECT)

In [ ]:
from huggingface_hub import hf_hub_download

DATA = Path('/content/data')
DATA.mkdir(parents=True, exist_ok=True)
SOURCE_REPO = 'pytorch-lifestream/retailhero-uplift'
def source_file(name):
    local = DATA / name
    if local.exists():
        return local
    return Path(hf_hub_download(repo_id=SOURCE_REPO,
        filename='data/'+name, repo_type='dataset'))

AB = source_file('uplift_train.csv.gz')
CLIENTS = source_file('clients.csv.gz')
RFM = DATA / 'final_stats.csv'
if not RFM.exists():
    print('RFM-файла пока нет — загружаем историю покупок (~609 МБ)...')
    purchases = source_file('purchases.csv.gz')
    subprocess.run([sys.executable, str(PROJECT/'scripts/build_rfm_features.py'),
        '--input', str(purchases), '--output', str(RFM)], check=True)
print('Готово:', AB, CLIENTS, RFM)

In [ ]:
RESULTS = Path('/content/retailhero_crossfit_results')
subprocess.run([sys.executable, str(PROJECT/'scripts/validate_crossfit.py'),
    '--ab',str(AB),'--clients',str(CLIENTS),'--rfm',str(RFM),
    '--output',str(RESULTS),'--bootstrap','2000'],check=True)

In [ ]:
import pandas as pd
from IPython.display import display
for filename in ('policy_summary.csv','bootstrap_intervals.csv','fold_results.csv'):
    print('\n', filename)
    display(pd.read_csv(RESULTS / filename).round(3))

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive_path=Path('/content/retailhero_crossfit_results.zip')
with ZipFile(archive_path,'w',ZIP_DEFLATED) as z:
    for path in RESULTS.iterdir():
        if path.name != 'oof_scored_clients.csv':
            z.write(path,arcname=path.name)
from google.colab import files
files.download(str(archive_path))